# Notebook 0 — Connect to Your Bedrock Agent from Code

Yesterday you built **TravelMind** in the Agent Builder console and chatted with it in the **Test** panel. Today you do the exact same thing **from Python**, on your own machine, in VS Code.

This is the on-ramp before Notebook 1. By the end you will:

- set up VS Code and confirm you are connected to AWS (from scratch)
- invoke your agent from code (the Test panel, as an API call)
- run **3 requests** — one simple, one multi-step, and one that fails on purpose
- read the **trace** (the console's *Show trace*, in code) and **debug + fix** a failure programmatically

Region throughout: **us-east-1**. Nothing here assumes prior Python-on-AWS experience.

## Yesterday (browser) vs today (code)

| In the console you... | In code that is... |
|---|---|
| opened the agent in Agent Builder | the **Agent ID** + alias in a config cell |
| typed in the **Test** panel | `invoke_agent(..., inputText=...)` |
| clicked **Show trace** | `enableTrace=True` and reading the trace |
| stayed in the same chat | reusing the same `sessionId` |
| clicked **Prepare** after edits | `prepare_agent(...)` (Notebook 3) |

Same agent, same behavior. You are swapping the mouse for a function call.

## Step 1 — Set up VS Code (from scratch)

Do these once, in a terminal, inside an empty project folder opened in VS Code:

1. **Install the VS Code extensions** *Python* and *Jupyter* (Extensions sidebar). Without *Jupyter* you cannot run `.ipynb` cells.
2. **Create and activate a virtual environment:**
   - `python -m venv .venv`
   - macOS/Linux: `source .venv/bin/activate`  ·  Windows (PowerShell): `.venv\Scripts\Activate.ps1`
3. **Install the libraries:** `pip install boto3 ipykernel`
4. **Give boto3 credentials** (pick one):
   - `aws configure` (needs the AWS CLI installed) — enter your key, secret, and `us-east-1`, **or**
   - set environment variables in the same terminal: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN` (if using temporary creds), and `AWS_DEFAULT_REGION=us-east-1`.
5. **Select the kernel:** open this notebook, click *Select Kernel* (top-right), and choose the `.venv` interpreter.

Then run the cell below to confirm the install.

In [ ]:
# If boto3 is missing, install it from a VS Code terminal (preferred) or uncomment this line:
# !pip install boto3 ipykernel

import boto3, botocore
print("boto3   :", boto3.__version__)       # need >= 1.34 for agents
print("botocore:", botocore.__version__)

## Step 2 — Credentials and region

You never put keys in code. `boto3` finds credentials automatically, in this order: **environment variables → `~/.aws/credentials` (written by `aws configure`) → an attached IAM role**. The cell below proves the chain works by asking AWS "who am I?" before you touch Bedrock.

*What changes in production:* there are no keys at all — the app runs under an **IAM role** (Lambda / ECS / EC2) and boto3 picks it up. Region comes from the environment, not a literal.

In [ ]:
REGION = "us-east-1"

sts = boto3.client("sts", region_name=REGION)
try:
    me = sts.get_caller_identity()
    print("Connected to AWS")
    print("  account :", me["Account"])
    print("  identity:", me["Arn"])
except botocore.exceptions.NoCredentialsError:
    print("No credentials found.")
    print("Fix: run `aws configure` in your VS Code terminal, or set AWS_ACCESS_KEY_ID / "
          "AWS_SECRET_ACCESS_KEY env vars, then restart the kernel.")

## Step 3 — The one client you need

There are several Bedrock clients (Notebook 1 covers all four). To **talk to an agent**, you need exactly one: **`bedrock-agent-runtime`**. This is the client behind the Test panel.

In [ ]:
agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)
print("agent runtime client ready")

## Step 4 — Find your Agent ID and alias

In the Bedrock console: **Agents → (your TravelMind agent) → Agent overview**. Copy the **Agent ID** (10 characters). The alias is **`TSTALIASID`** — the always-present test alias that the Test panel itself uses, which points at your latest draft.

Paste the Agent ID below. The cell stops you with a clear message if you forget.

In [ ]:
AGENT_ID = "XXXXXXXXXX"     # <-- paste your 10-char Agent ID from the console overview
ALIAS_ID = "TSTALIASID"     # the test alias (what the Test panel uses)

if AGENT_ID == "XXXXXXXXXX":
    print("ACTION NEEDED: paste your real Agent ID into AGENT_ID above, then re-run.")
else:
    print("config set -> agent", AGENT_ID, "| alias", ALIAS_ID)

## Step 5 — Invoke the agent (the Test panel, in code)

`invoke_agent` is the API behind the Test panel. Two things trip up beginners:

- The reply comes back on the key **`completion`**, which is an **event stream** you loop over — not a string you read directly.
- **`sessionId`** is the conversation handle. Reuse the same id to continue a chat; use a new id to start fresh.

The helper below wraps all of that. We test it with a plain hello to confirm the wire is live.

In [ ]:
import uuid, json

def ask_agent(prompt, session_id=None, trace=False):
    """Send one message to the agent. Returns (answer, traces, session_id).
    This is the code version of typing into the Agent Builder 'Test' panel."""
    session_id = session_id or str(uuid.uuid4())
    response = agent_runtime.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=ALIAS_ID,
        sessionId=session_id,     # same id  ->  same conversation
        inputText=prompt,
        enableTrace=trace,        # the 'Show trace' toggle, in code
    )
    answer, traces = "", []
    for event in response["completion"]:        # completion is a STREAM
        if "chunk" in event:                    # a piece of the answer
            answer += event["chunk"]["bytes"].decode("utf-8")
        elif "trace" in event:                  # a step of reasoning (only if enableTrace=True)
            traces.append(event["trace"]["trace"])
    return answer, traces, session_id

hello, _, _ = ask_agent("Hello, what can you help me with?")
print(hello)

## Happy path 1 — simple (one tool, one answer)

A direct question your agent can answer with a single lookup. Use a PNR your agent's tools actually know (we use `ABC123` from yesterday's demo — swap in yours if different).

In [ ]:
answer, _, _ = ask_agent("What is the status of booking ABC123?")
print(answer)

## Reading the trace — the console's *Show trace*, in code

When you ticked *Show trace* yesterday, you saw the agent's steps. In code those steps arrive as `trace` events. The piece you care about is the **orchestration trace**, which has three parts:

- **rationale** — the model **thinking**
- **invocationInput** — the model **acting** (which tool, with what input)
- **observation** — the **result** coming back, and the **final answer**

A failure (if one happens) shows up as a separate **`failureTrace`**. The helper below prints all of it cleanly.

In [ ]:
def print_trace(traces):
    for t in traces:
        if "failureTrace" in t:                      # the agent hit an error
            print("FAILURE :", t["failureTrace"].get("failureReason", "")[:300])
        ot = t.get("orchestrationTrace")
        if not ot:
            continue
        if "rationale" in ot:                        # THINK
            print("THINK   :", ot["rationale"]["text"][:240])
        if "invocationInput" in ot:                  # ACT
            agi = ot["invocationInput"].get("actionGroupInvocationInput")
            if agi:
                print("ACT     :", agi.get("function"), "->", agi.get("parameters"))
        if "observation" in ot:                      # OBSERVE / FINAL
            obs = ot["observation"]
            if obs.get("actionGroupInvocationOutput"):
                print("OBSERVE :", obs["actionGroupInvocationOutput"].get("text", "")[:240])
            if obs.get("finalResponse"):
                print("FINAL   :", obs["finalResponse"]["text"][:280])
        print("-" * 60)

answer, traces, _ = ask_agent("What is the status of booking ABC123?", trace=True)
print_trace(traces)
print("\nANSWER:", answer)

## Happy path 2 — intermediate (multiple steps)

A request that needs more than one tool. The agent should look up the booking, find why it was disrupted, then fetch alternatives — you will see several **ACT / OBSERVE** pairs chain together in the trace before the final answer.

In [ ]:
answer, traces, _ = ask_agent(
    "My flight on PNR ABC123 was disrupted. Why, and what are my rebooking options?",
    trace=True,
)
print_trace(traces)
print("\nANSWER:", answer)

## Fail path — make it fail, then debug it

Now a request that does **not** go well: ask about a booking that does not exist. To a user this looks like "the agent is broken." Your job is to read the trace and find out *what actually failed* — the model, the tool, or the input.

In [ ]:
bad_answer, bad_traces, _ = ask_agent(
    "What is the status of booking ZZ000?",     # ZZ000 is not a real booking
    trace=True,
)
print_trace(bad_traces)
print("\nANSWER:", bad_answer)

## Debug from the trace, then fix in code

Read the trace above. The pattern you will see is:

- **ACT** `lookup_booking -> {'pnr': 'ZZ000'}` — the agent did the right thing
- **OBSERVE** `{"found": false, ...}` — the **tool** returned "not found"
- **FINAL** — the agent honestly says it cannot find the booking

So this is **not** an agent bug and **not** a code bug. It is an **input** problem: the PNR does not exist (or was mistyped). That is the whole point of the trace — it tells you *which layer* failed so you do not waste time fixing the wrong one.

**The programmatic fix:** validate and normalize the PNR at your boundary *before* spending an agent call. Catch obviously-bad input early, give the user a fast, clear message, and normalize messy input (lowercase, spaces, hyphens) so a valid booking is not missed over formatting.

In [ ]:
import re

def normalize_pnr(text):
    # a PNR is 6 alphanumerics; strip case, spaces, and stray punctuation
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

def looks_like_pnr(pnr):
    return bool(re.match(r"^[A-Z0-9]{6}$", pnr))

def ask_about_booking(raw_pnr, session_id=None):
    pnr = normalize_pnr(raw_pnr)
    if not looks_like_pnr(pnr):
        # fail fast, in YOUR code, before calling the agent
        return f"'{raw_pnr}' is not a valid PNR (need 6 letters/numbers). Please check and resend."
    answer, _, _ = ask_agent(f"What is the status of booking {pnr}?", session_id=session_id)
    return answer

# messy but valid input is normalized and now succeeds:
print("messy input 'abc 123' ->", normalize_pnr("abc 123"))
print(ask_about_booking("abc 123"))      # becomes ABC123 -> real lookup

# genuinely bad input is caught before any agent call:
print()
print(ask_about_booking("nope!"))

## When the SDK itself errors (connection-level)

The failure above came from the agent's *answer*. A different class of failure stops you before you even get an answer — a `ClientError` from the SDK. These are the ones you will hit first while wiring up on VS Code. Decode the error code, apply the fix.

| Error code | What it usually means | Fix |
|---|---|---|
| `resourceNotFoundException` | wrong Agent ID / alias / region | re-copy the Agent ID from *Agent overview*; alias is `TSTALIASID`; region `us-east-1` |
| `accessDeniedException` | IAM lacks `bedrock:InvokeAgent`, or model access not enabled | add the permission / enable model access in this region |
| `validationException` | malformed request (empty `inputText`, bad `sessionId`) | check your parameters |
| `throttlingException` | too many requests | retry with backoff (Notebook 5) |
| `dependencyFailedException` | the agent's **action-group Lambda** errored | read **that Lambda's** CloudWatch logs — the bug is in your tool code |
| `serviceQuotaExceededException` | you hit a quota | request an increase in Service Quotas |

In [ ]:
from botocore.exceptions import ClientError

FIXES = {
    "resourcenotfoundexception": "Wrong AGENT_ID / ALIAS_ID / region. Re-copy from Agents > overview; alias = TSTALIASID; region = us-east-1.",
    "accessdeniedexception":     "IAM is missing bedrock:InvokeAgent, or model access is not enabled in this region.",
    "validationexception":       "Malformed request (empty inputText, bad sessionId, etc.). Check your parameters.",
    "throttlingexception":       "Too many requests. Retry with exponential backoff (see Notebook 5).",
    "dependencyfailedexception": "The agent's action-group Lambda raised an error. Read THAT Lambda's CloudWatch logs.",
    "servicequotaexceededexception": "You hit a service quota. Request an increase in the Service Quotas console.",
}

def explain_bedrock_error(e):
    code = e.response["Error"]["Code"]
    print("error code :", code)
    print("message    :", e.response["Error"]["Message"][:200])
    print("likely fix :", FIXES.get(code.lower(), "Read the message above and the Bedrock docs."))

# Demonstrate: call with a deliberately wrong alias and decode the error (nothing is changed in AWS).
try:
    agent_runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId="WRONGALIAS",
        sessionId="debug-demo", inputText="hi",
    )
except ClientError as e:
    explain_bedrock_error(e)

## Recap

You reproduced the Agent Builder Test panel in code:

- connected to AWS from VS Code (`aws configure` + `get_caller_identity`)
- invoked the agent with one client, `bedrock-agent-runtime`
- ran a **simple**, an **intermediate**, and a **failing** request
- read the **trace** (THINK / ACT / OBSERVE) and used it to locate a failure **at the right layer**, then fixed it in code with input validation
- learned the common **SDK errors** and their fixes

**Next — Notebook 1** goes one level deeper: the four Bedrock clients, the Converse API, and the model on its own (text in, text out) before you rebuild the agent loop by hand in Notebook 2.